In [1]:
import zipfile

with zipfile.ZipFile("/content/reviews_data_dump.zip",'r') as zip_ref:
  zip_ref.extractall("/content/data")

In [2]:
import os

os.listdir("/content/data")

['reviews_badminton', 'reviews_tawa', 'reviews_tea']

In [3]:
os.listdir('/content/data/reviews_badminton')

['credits.txt', 'data.csv']

In [4]:
import pandas as pd

df = pd.read_csv('/content/data/reviews_badminton/data.csv')

In [7]:
df.head()

,Reviewer Name,Review Title,Place of Review,Up Votes,Down Votes,Month,Review text,Ratings
0,Kamal Suresh,Nice product,"Certified Buyer, Chirakkal",889.0,64.0,Feb 2021,"Nice product, good quality, but price is now r...",4
1,Flipkart Customer,Don't waste your money,"Certified Buyer, Hyderabad",109.0,6.0,Feb 2021,They didn't supplied Yonex Mavis 350. Outside ...,1
2,A. S. Raja Srinivasan,Did not meet expectations,"Certified Buyer, Dharmapuri",42.0,3.0,Apr 2021,Worst product. Damaged shuttlecocks packed in ...,1
3,Suresh Narayanasamy,Fair,"Certified Buyer, Chennai",25.0,1.0,NaN,"Quite O. K. , but nowadays the quality of the...",3
4,ASHIK P A,Over priced,NaN,147.0,24.0,Apr 2016,Over pricedJust â?¹620 ..from retailer.I didn'...,1


In [6]:
df.columns

Index(['Reviewer Name', 'Review Title', 'Place of Review', 'Up Votes',
       'Down Votes', 'Month', 'Review text', 'Ratings'],
      dtype='object')

In [8]:
df.shape

(8518, 8)

In [9]:
df.isnull().sum()

,0
Reviewer Name,10
Review Title,10
Place of Review,50
Up Votes,10
Down Votes,10
Month,465
Review text,8
Ratings,0


In [10]:
df = df[['Review text','Ratings']]

In [11]:
df.head()

,Review text,Ratings
0,"Nice product, good quality, but price is now r...",4
1,They didn't supplied Yonex Mavis 350. Outside ...,1
2,Worst product. Damaged shuttlecocks packed in ...,1
3,"Quite O. K. , but nowadays the quality of the...",3
4,Over pricedJust â?¹620 ..from retailer.I didn'...,1


In [12]:
def sentiment_label(rating):

    if rating >= 4:
        return 1

    elif rating <= 2:
        return 0

    else:
        return None

In [13]:
df['sentiment'] = df['Ratings'].apply(sentiment_label)

In [14]:
df = df.dropna(subset=['sentiment'])

In [15]:
df['sentiment'].value_counts()

,count
sentiment,
1.0,6826
0.0,1077


In [16]:
df.isnull().sum()

,0
Review text,8
Ratings,0
sentiment,0


In [17]:
df = df.dropna(subset=['Review text'])

In [18]:
df.shape

(7895, 3)

In [20]:
import nltk

nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [21]:
import re

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [22]:
lemmatizer = WordNetLemmatizer()

stop_words = set(stopwords.words('english'))

In [23]:
def clean_text(text):

    text = str(text).lower()

    # remove urls
    text = re.sub(r'http\S+', '', text)

    # keep only alphabets
    text = re.sub(r'[^a-zA-Z]', ' ', text)

    # split words
    words = text.split()

    # remove stopwords + lemmatization
    words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stop_words
    ]

    return " ".join(words)

In [24]:
df['clean_review'] = df['Review text'].apply(clean_text)

In [25]:
df[['Review text', 'clean_review']].head()

,Review text,clean_review
0,"Nice product, good quality, but price is now r...",nice product good quality price rising bad sig...
1,They didn't supplied Yonex Mavis 350. Outside ...,supplied yonex mavis outside cover yonex ad in...
2,Worst product. Damaged shuttlecocks packed in ...,worst product damaged shuttlecock packed new b...
4,Over pricedJust â?¹620 ..from retailer.I didn'...,pricedjust retailer understand wat advantage b...
5,Good quality product. Delivered on time.READ MORE,good quality product delivered time read


In [26]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [27]:
tfidf = TfidfVectorizer(max_features=5000)

In [28]:
X = tfidf.fit_transform(df['clean_review']).toarray()

In [29]:
y = df['sentiment']

In [30]:
from sklearn.model_selection import train_test_split

In [31]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [32]:
print(X_train.shape)
print(X_test.shape)

(6316, 2882)
(1579, 2882)


In [33]:
from sklearn.linear_model import LogisticRegression

In [34]:
model = LogisticRegression()
model

LogisticRegression()

In [35]:
model.fit(X_train, y_train)

LogisticRegression()

In [36]:
y_pred = model.predict(X_test)

In [37]:
y_pred

array([1., 1., 0., ..., 1., 1., 1.])

In [38]:
from sklearn.metrics import classification_report
from sklearn.metrics import f1_score

In [39]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

         0.0       0.86      0.50      0.63       207
         1.0       0.93      0.99      0.96      1372

    accuracy                           0.92      1579
   macro avg       0.89      0.74      0.79      1579
weighted avg       0.92      0.92      0.91      1579



In [72]:
log_f1_score = f1_score(y_test, y_pred)
log_f1_score

0.9572589191098552

In [40]:
print("F1 Score:", f1_score(y_test, y_pred))

F1 Score: 0.9572589191098552


## Naive Bayes

In [44]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import f1_score

In [45]:
nb_model = MultinomialNB()

In [46]:
nb_model.fit(X_train, y_train)

MultinomialNB()

In [48]:
nb_pred = nb_model.predict(X_test)

In [73]:
nb_f1_score = f1_score(y_test, nb_pred)
nb_f1_score

0.9492703266157053

In [49]:
print("Naive Bayes F1 Score:",
      f1_score(y_test, nb_pred))

Naive Bayes F1 Score: 0.9492703266157053


## Random Forest

In [50]:
from sklearn.ensemble import RandomForestClassifier

In [52]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)
rf_model

RandomForestClassifier(random_state=42)

In [53]:
rf_model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [55]:
rf_pred = rf_model.predict(X_test)

In [74]:
rf_f1_score = f1_score(y_test, rf_pred)
rf_f1_score

0.9569548203486303

In [56]:
print("Random Forest F1 Score:",
      f1_score(y_test, rf_pred))

Random Forest F1 Score: 0.9569548203486303


## SVM

In [57]:
from sklearn.svm import LinearSVC

In [58]:
svm_model = LinearSVC()

In [59]:
svm_model.fit(X_train, y_train)

LinearSVC()

In [60]:
svm_pred = svm_model.predict(X_test)

In [75]:
svm_f1_score = f1_score(y_test, svm_pred)
svm_f1_score

0.9620615604867573

In [61]:
print("SVM F1 Score:",
      f1_score(y_test, svm_pred))

SVM F1 Score: 0.9620615604867573


## XG Boost

In [62]:
!pip install xgboost

In [63]:
from xgboost import XGBClassifier

In [64]:
xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42
)

In [65]:
xgb_model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=100,
              n_jobs=None, num_parallel_tree=None, ...)

In [66]:
xgb_pred = xgb_model.predict(X_test)

In [76]:
xgb_f1_score = f1_score(y_test, xgb_pred)
xgb_f1_score

0.955595026642984

In [67]:
print("XGBoost F1 Score:",
      f1_score(y_test, xgb_pred))

XGBoost F1 Score: 0.955595026642984


In [79]:
results = {
    "Model": [
        "Logistic Regression",
        "Naive Bayes",
        "Random Forest",
        "SVM",
        "XGBoost"
    ],

    "F1 Score": [
        log_f1_score,
        nb_f1_score,
        rf_f1_score,
        svm_f1_score,
        xgb_f1_score
    ]
}
results

{'Model': ['Logistic Regression',
  'Naive Bayes',
  'Random Forest',
  'SVM',
  'XGBoost'],
 'F1 Score': [0.9572589191098552,
  0.9492703266157053,
  0.9569548203486303,
  0.9620615604867573,
  0.955595026642984]}

In [80]:
results_df = pd.DataFrame(results)

results_df

,Model,F1 Score
0,Logistic Regression,0.957259
1,Naive Bayes,0.949270
2,Random Forest,0.956955
3,SVM,0.962062
4,XGBoost,0.955595


In [68]:
from sklearn.model_selection import GridSearchCV

In [69]:
param_grid = {
    'C': [0.1, 1, 10],
    'max_iter': [1000, 2000]
}

In [70]:
grid_search = GridSearchCV(
    LinearSVC(),
    param_grid,
    cv=3,
    scoring='f1',
    n_jobs=-1
)

In [71]:
grid_search.fit(X_train, y_train)

GridSearchCV(cv=3, estimator=LinearSVC(), n_jobs=-1,
             param_grid={'C': [0.1, 1, 10], 'max_iter': [1000, 2000]},
             scoring='f1')

In [81]:
print(grid_search.best_params_)

{'C': 1, 'max_iter': 1000}


In [82]:
best_svm = grid_search.best_estimator_

In [83]:
tuned_pred = best_svm.predict(X_test)

In [84]:
print("Tuned SVM F1 Score:",
      f1_score(y_test, tuned_pred))

Tuned SVM F1 Score: 0.9620615604867573


In [87]:
import pickle

In [88]:
pickle.dump(best_svm,
            open('svm_sentiment_model.pkl', 'wb'))

In [89]:
pickle.dump(tfidf,
            open('tfidf_vectorizer.pkl', 'wb'))